
# 🚴 Dashboard Calidad del Aire — Bogotá (RMCAB)

Notebook completo y reproducible: carga el reporte de la RMCAB, **limpia los datos**
(elimina las horas sin ningún dato reportado), calcula estadísticas por estación,
por hora del día y por tipo de día (Entre Semana / Fin de Semana), y genera los
5 gráficos interactivos de Plotly + el HTML final del dashboard.

**Requisito:** tener el archivo `reporteIboca__1___1_.xls` (el reporte exportado
desde el visor de la RMCAB) en la misma carpeta que este notebook, o ajustar la
variable `RAW_PATH` en la siguiente celda.

**Librerías necesarias:** `pandas`, `numpy`, `plotly` (`pip install pandas numpy plotly`).


## 1. Configuración e importación de librerías

In [ ]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pickle

pd.set_option('future.no_silent_downcasting', True)

# Ruta del archivo de datos crudo (reporte exportado de la RMCAB).
# Ajusta esta ruta si el archivo está en otra carpeta.
RAW_PATH = "reporteIboca__1___1_.xls"

# Ruta de salida del dashboard final
OUTPUT_HTML = "dashboard_aire_bogota.html"



## 2. Carga y limpieza de datos

El archivo `.xls` en realidad es una tabla HTML exportada por el visor de la RMCAB.
Contiene:
- Una fila de encabezado por estación con las columnas `Concentración`, `NowCast`, `IBOCA`.
- Filas de resumen al final (`AVG`, `Num`, `Datos [%]`) que no son datos horarios reales.
- Celdas con el texto literal `"Sin data"` cuando el sensor no reportó nada esa hora.

**Regla de limpieza clave (pedida explícitamente):** si en una marca de tiempo
**ninguna** estación reportó dato, esa hora se elimina por completo del conjunto de
datos — no debe aparecer ni en gráficos ni en tablas.


In [ ]:

tables = pd.read_html(RAW_PATH)
meta_table, df = tables[0], tables[1]

def fix_encoding(s):
    '''Corrige doble-codificación latin1->utf8 en nombres de estaciones (tildes, ñ).'''
    if isinstance(s, str):
        try:
            return s.encode('latin1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return s
    return s

# Normaliza encabezados (multi-índice: Estación / Variable / Métrica)
new_cols = []
for c in df.columns:
    t = tuple(fix_encoding(x) for x in c)
    if t[0] == 'Unnamed: 0_level_0':
        t = ('Fecha', 'Fecha', 'Fecha')
    new_cols.append(t)
df.columns = pd.MultiIndex.from_tuples(new_cols)
fecha_col = ('Fecha', 'Fecha', 'Fecha')

# Elimina filas de resumen del reporte (AVG / Num / Datos [%])
df = df[~df[fecha_col].isin(['AVG', 'Num', 'Datos [%]'])].copy()
df[fecha_col] = pd.to_datetime(df[fecha_col])

n_total_horas = len(df)
print(f"Marcas de tiempo totales en el reporte: {n_total_horas}")


In [ ]:

# --- Detecta y elimina horas donde NINGUNA estación reportó dato ---
iboca_cols = [c for c in df.columns if c[2] == 'IBOCA']
iboca_raw = df[iboca_cols].replace('Sin data', np.nan).apply(pd.to_numeric, errors='coerce')
horas_sin_datos_mask = iboca_raw.isna().all(axis=1)
n_horas_sin_datos = int(horas_sin_datos_mask.sum())

print(f"Horas completamente sin datos (se eliminan): {n_horas_sin_datos} de {n_total_horas} "
      f"({n_horas_sin_datos/n_total_horas*100:.1f}%)")

# Elimina esas horas del dataframe base ANTES de construir cualquier tabla o gráfico
df_limpio = df.loc[~horas_sin_datos_mask].copy()


In [ ]:

stations = sorted(set(c[0] for c in df.columns if c[0] != 'Fecha'))
print(f"Estaciones ({len(stations)}):", stations)

# --- Convierte a formato largo: una fila por estación x timestamp ---
records = []
for st in stations:
    sub = df_limpio[[fecha_col,
                      (st, 'PM2.5 µg/m3', 'Concentración'),
                      (st, 'PM2.5 µg/m3', 'NowCast'),
                      (st, 'PM2.5 µg/m3', 'IBOCA')]].copy()
    sub.columns = ['Fecha', 'Concentracion', 'NowCast', 'IBOCA']
    sub['Estacion'] = st
    records.append(sub)
long_df = pd.concat(records, ignore_index=True)

for c in ['Concentracion', 'NowCast', 'IBOCA']:
    long_df[c] = long_df[c].replace('Sin data', np.nan)
    long_df[c] = pd.to_numeric(long_df[c], errors='coerce')

n_registros_totales = len(long_df)
valid = long_df.dropna(subset=['IBOCA']).copy()
n_registros_validos = len(valid)
print(f"Registros estación-hora: {n_registros_totales} totales -> {n_registros_validos} válidos "
      f"({n_registros_validos/n_registros_totales*100:.1f}%)")


## 3. Variables derivadas (hora, tipo de día, mes, categoría de riesgo)

In [ ]:

valid['Hora'] = valid['Fecha'].dt.hour
valid['DiaSemana'] = valid['Fecha'].dt.dayofweek
valid['Tipo_Dia'] = np.where(valid['DiaSemana'] >= 5, 'Fin de Semana', 'Entre Semana')
valid['Fecha_Dia'] = valid['Fecha'].dt.floor('D')
valid['Mes'] = valid['Fecha'].dt.to_period('M').astype(str)

def categoria_riesgo(v):
    '''Categoría de riesgo según el índice oficial IBOCA (Secretaría de Ambiente de Bogotá).'''
    if v <= 50:
        return 'Buena'
    elif v <= 100:
        return 'Moderada'
    elif v <= 150:
        return 'Dañina (grupos sensibles)'
    elif v <= 200:
        return 'Dañina'
    elif v <= 300:
        return 'Muy Dañina'
    else:
        return 'Peligrosa'

valid['Categoria_Riesgo'] = valid['IBOCA'].apply(categoria_riesgo)

CAT_ORDER = ['Buena', 'Moderada', 'Dañina (grupos sensibles)', 'Dañina', 'Muy Dañina', 'Peligrosa']
CAT_COLOR = {
    'Buena': '#4ADE80',
    'Moderada': '#FDE047',
    'Dañina (grupos sensibles)': '#FB923C',
    'Dañina': '#F87171',
    'Muy Dañina': '#C084FC',
    'Peligrosa': '#7F1D1D',
}

valid.head()



## 4. Coordenadas de las estaciones (para el mapa)

Coordenadas aproximadas de las 19 estaciones activas de la RMCAB presentes en este reporte.


In [ ]:

COORDS = {
    'Usaquen': (4.7110, -74.0301),
    'Suba': (4.7590, -74.0938),
    'Guaymaral': (4.7909, -74.0435),
    'Las Ferias': (4.6883, -74.0776),
    'Colina': (4.7286, -74.0454),
    'Bolivia': (4.7423, -74.1145),
    'MinAmbiente': (4.6260, -74.0665),
    'CDAR': (4.6580, -74.0840),
    'Carvajal - Sevillana': (4.6167, -74.1494),
    'Puente Aranda': (4.6156, -74.1114),
    'Kennedy': (4.6280, -74.1518),
    'Tunal': (4.5764, -74.1300),
    'Ciudad Bolívar': (4.5495, -74.1650),
    'San Cristobal': (4.5573, -74.0837),
    'Usme': (4.5000, -74.1264),
    'Jazmín': (4.6046, -74.1013),
    'Fontibon': (4.6742, -74.1469),
    'Móvil Fontibón': (4.6742, -74.1469),
    'Móvil 7ma': (4.6270, -74.0637),
}



## 5. Agregaciones para cada gráfico

Métrica principal: **PM2.5** (µg/m³). La categoría de riesgo (colores) se conserva
calculada sobre el índice oficial **IBOCA**.

Como salvaguarda adicional, cualquier agrupación que quede con 0 registros
(`count == 0`) se descarta antes de graficar — no debería ocurrir tras la limpieza,
pero se deja como control defensivo.


In [ ]:

def drop_empty_groups(df_group, count_col='count'):
    return df_group[df_group[count_col] > 0].copy()

# --- 5.1 Comparativa por estación ---
est_stats = valid.groupby('Estacion').agg(
    PM25_prom=('Concentracion', 'mean'),
    PM25_std=('Concentracion', 'std'),
    IBOCA_prom=('IBOCA', 'mean'),
    count=('IBOCA', 'count'),
).reset_index()
est_stats = drop_empty_groups(est_stats)
est_stats['Categoria'] = est_stats['IBOCA_prom'].apply(categoria_riesgo)
est_stats = est_stats.sort_values('PM25_prom')
est_stats


In [ ]:

# --- 5.2 Perfil horario (Entre Semana vs Fin de Semana) ---
hora_stats = valid.groupby(['Tipo_Dia', 'Hora']).agg(
    PM25_prom=('Concentracion', 'mean'),
    IBOCA_prom=('IBOCA', 'mean'),
    count=('IBOCA', 'count'),
).reset_index()
hora_stats = drop_empty_groups(hora_stats)
# Solo se muestran las horas que tienen datos para AMBOS tipos de día
horas_validas = sorted(set(hora_stats.loc[hora_stats['Tipo_Dia'] == 'Entre Semana', 'Hora'])
                        & set(hora_stats.loc[hora_stats['Tipo_Dia'] == 'Fin de Semana', 'Hora']))
hora_stats = hora_stats[hora_stats['Hora'].isin(horas_validas)]
hora_stats.head()


In [ ]:

# --- 5.3 Comparativa por estación: Entre Semana vs Fin de Semana ---
est_tipodia = valid.groupby(['Estacion', 'Tipo_Dia']).agg(
    PM25_prom=('Concentracion', 'mean'),
    IBOCA_prom=('IBOCA', 'mean'),
    count=('Concentracion', 'count'),
).reset_index()
est_tipodia = drop_empty_groups(est_tipodia)
est_tipodia['Categoria'] = est_tipodia['IBOCA_prom'].apply(categoria_riesgo)

# Solo estaciones con dato válido en AMBOS tipos de día
estaciones_completas = sorted(
    set(est_tipodia.loc[est_tipodia['Tipo_Dia'] == 'Entre Semana', 'Estacion'])
    & set(est_tipodia.loc[est_tipodia['Tipo_Dia'] == 'Fin de Semana', 'Estacion'])
)
est_tipodia = est_tipodia[est_tipodia['Estacion'].isin(estaciones_completas)]
orden_estaciones = est_stats[est_stats['Estacion'].isin(estaciones_completas)]['Estacion'].tolist()
est_tipodia.head()


In [ ]:

# --- 5.4 Tendencia temporal (promedio diario de la ciudad) ---
diaria = valid.groupby('Fecha_Dia').agg(
    PM25_prom=('Concentracion', 'mean'),
    IBOCA_prom=('IBOCA', 'mean'),
    count=('IBOCA', 'count'),
).reset_index()
diaria = drop_empty_groups(diaria)
diaria = diaria.sort_values('Fecha_Dia')
diaria['PM25_MM7'] = diaria['PM25_prom'].rolling(7, min_periods=1).mean()
diaria.head()


In [ ]:

# --- 5.5 Datos geográficos para el mapa ---
mapa_df = est_stats.copy()
mapa_df['lat'] = mapa_df['Estacion'].map(lambda s: COORDS.get(s, (np.nan, np.nan))[0])
mapa_df['lon'] = mapa_df['Estacion'].map(lambda s: COORDS.get(s, (np.nan, np.nan))[1])
mapa_df = mapa_df.dropna(subset=['lat', 'lon'])

mapa_tipodia = est_tipodia.copy()
mapa_tipodia['lat'] = mapa_tipodia['Estacion'].map(lambda s: COORDS.get(s, (np.nan, np.nan))[0])
mapa_tipodia['lon'] = mapa_tipodia['Estacion'].map(lambda s: COORDS.get(s, (np.nan, np.nan))[1])
mapa_tipodia = mapa_tipodia.dropna(subset=['lat', 'lon'])


## 6. KPIs generales

In [ ]:

kpi = {
    'pm25_prom': valid['Concentracion'].mean(),
    'iboca_prom': valid['IBOCA'].mean(),
    'estacion_mas_limpia': est_stats.iloc[0]['Estacion'],
    'estacion_mas_limpia_val': est_stats.iloc[0]['PM25_prom'],
    'estacion_mas_contaminada': est_stats.iloc[-1]['Estacion'],
    'estacion_mas_contaminada_val': est_stats.iloc[-1]['PM25_prom'],
    'pct_validos': n_registros_validos / n_registros_totales * 100,
    'n_horas_sin_datos': n_horas_sin_datos,
    'n_total_horas': n_total_horas,
    'n_estaciones': len(stations),
    'n_dias': valid['Fecha_Dia'].nunique(),
    'fecha_min': valid['Fecha'].min(),
    'fecha_max': valid['Fecha'].max(),
    'entre_semana_n': int((valid['Tipo_Dia'] == 'Entre Semana').sum()),
    'fin_semana_n': int((valid['Tipo_Dia'] == 'Fin de Semana').sum()),
    'categoria_predominante': valid['Categoria_Riesgo'].mode().iloc[0],
}
kpi



## 7. Gráficos Plotly

Se generan los 5 gráficos del dashboard. En el notebook cada `fig.show()` los
despliega interactivos directamente en la celda de salida.


In [ ]:

PLOT_BG = "#FFFFFF"
GRID_COLOR = "#E9EDF2"
FONT = dict(family="'Inter','Segoe UI',sans-serif", color="#22303F", size=13)
LIMITE_PM25_24H = 37  # Resolución 2254 de 2017 (MinAmbiente) — límite diario PM2.5

def base_layout(title, height=440):
    return dict(
        title=dict(text=title, font=dict(size=17, color="#0F1B24"), x=0.02, xanchor="left"),
        plot_bgcolor=PLOT_BG, paper_bgcolor=PLOT_BG, font=FONT,
        margin=dict(l=60, r=30, t=60, b=50), height=height,
        hoverlabel=dict(bgcolor="white", font_size=12, bordercolor="#D8DEE6"),
    )


### 7.1 Comparativa de PM2.5 por estación

In [ ]:

colors_1 = [CAT_COLOR[c] for c in est_stats['Categoria']]
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    y=est_stats['Estacion'], x=est_stats['PM25_prom'], orientation='h',
    marker=dict(color=colors_1, line=dict(color="rgba(0,0,0,0.15)", width=1)),
    error_x=dict(type='data', array=est_stats['PM25_std'], color='rgba(0,0,0,0.25)', thickness=1),
    customdata=np.stack([est_stats['Categoria'], est_stats['count'], est_stats['IBOCA_prom'].round(1)], axis=-1),
    hovertemplate="<b>%{y}</b><br>PM2.5 promedio: %{x:.1f} µg/m³<br>Categoría (IBOCA): %{customdata[0]}"
                  "<br>IBOCA promedio: %{customdata[2]}<br>Registros válidos: %{customdata[1]}<extra></extra>",
))
fig1.add_vline(x=LIMITE_PM25_24H, line=dict(color="rgba(0,0,0,0.35)", dash="dot", width=1.3),
               annotation_text=f"Límite diario Colombia ({LIMITE_PM25_24H} µg/m³)",
               annotation_font_size=11, annotation_position="top")
fig1.update_layout(**base_layout("Comparativa de PM2.5 promedio por estación", height=560))
fig1.update_xaxes(title_text="PM2.5 (µg/m³) — promedio ± desv. estándar", gridcolor=GRID_COLOR)
fig1.update_yaxes(gridcolor=GRID_COLOR)
fig1.show()


### 7.2 Perfil horario — Entre Semana vs Fin de Semana

In [ ]:

line_colors = {"Entre Semana": "#2563EB", "Fin de Semana": "#F97316"}
fig2 = go.Figure()
for tipo in ["Entre Semana", "Fin de Semana"]:
    sub = hora_stats[hora_stats['Tipo_Dia'] == tipo].sort_values('Hora')
    fig2.add_trace(go.Scatter(
        x=sub['Hora'], y=sub['PM25_prom'], mode='lines+markers', name=tipo,
        line=dict(color=line_colors[tipo], width=3), marker=dict(size=6),
        customdata=np.stack([sub['count'], sub['IBOCA_prom'].round(1)], axis=-1),
        hovertemplate=f"<b>{tipo}</b><br>Hora: " + "%{x}:00<br>PM2.5 promedio: %{y:.1f} µg/m³"
                      "<br>IBOCA promedio: %{customdata[1]}<br>Registros válidos: %{customdata[0]}<extra></extra>",
    ))
fig2.update_layout(**base_layout("Perfil horario de PM2.5 — Entre Semana vs Fin de Semana"))
fig2.update_xaxes(title_text="Hora del día", dtick=1, gridcolor=GRID_COLOR)
fig2.update_yaxes(title_text="PM2.5 promedio (µg/m³)", gridcolor=GRID_COLOR)
fig2.update_layout(legend=dict(orientation="h", y=1.08, x=0.02, xanchor="left"))
fig2.show()


### 7.3 Tendencia temporal — PM2.5 promedio diario

In [ ]:

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=diaria['Fecha_Dia'], y=diaria['PM25_prom'], mode='markers', name='Promedio diario',
    marker=dict(size=4, color="rgba(37,99,235,0.35)"),
    hovertemplate="%{x|%d %b %Y}<br>PM2.5 promedio diario: %{y:.1f} µg/m³<extra></extra>",
))
fig3.add_trace(go.Scatter(
    x=diaria['Fecha_Dia'], y=diaria['PM25_MM7'], mode='lines', name='Media móvil (7 días)',
    line=dict(color="#1D4ED8", width=3),
    hovertemplate="%{x|%d %b %Y}<br>Media móvil 7d: %{y:.1f} µg/m³<extra></extra>",
))
fig3.add_hline(y=LIMITE_PM25_24H, line=dict(color="#DC2626", dash="dash", width=1.5),
               annotation_text=f"Límite diario Colombia ({LIMITE_PM25_24H} µg/m³)",
               annotation_position="top left", annotation_font_size=11)
fig3.update_layout(**base_layout("Tendencia temporal — PM2.5 promedio diario de la ciudad"))
fig3.update_xaxes(title_text="Fecha", gridcolor=GRID_COLOR)
fig3.update_yaxes(title_text="PM2.5 promedio diario (µg/m³)", gridcolor=GRID_COLOR)
fig3.update_layout(legend=dict(orientation="h", y=1.08, x=0.02, xanchor="left"))
fig3.show()


### 7.4 Comparativa por estación: Entre Semana vs Fin de Semana

In [ ]:

fig5 = go.Figure()
for tipo in ["Entre Semana", "Fin de Semana"]:
    sub = est_tipodia[est_tipodia['Tipo_Dia'] == tipo].set_index('Estacion').reindex(orden_estaciones).reset_index()
    fig5.add_trace(go.Bar(
        x=sub['Estacion'], y=sub['PM25_prom'], name=tipo, marker=dict(color=line_colors[tipo]),
        customdata=sub['count'],
        hovertemplate=f"<b>%{{x}}</b><br>{tipo}<br>PM2.5 promedio: " + "%{y:.1f} µg/m³"
                      "<br>Registros válidos: %{customdata}<extra></extra>",
    ))
fig5.add_hline(y=LIMITE_PM25_24H, line=dict(color="#DC2626", dash="dash", width=1.3),
               annotation_text=f"Límite diario ({LIMITE_PM25_24H} µg/m³)", annotation_font_size=11,
               annotation_position="top left")
fig5.update_layout(**base_layout("Comparativa por estación: Entre Semana vs Fin de Semana (PM2.5)", height=480))
fig5.update_xaxes(title_text="Estación", gridcolor=GRID_COLOR, tickangle=-35)
fig5.update_yaxes(title_text="PM2.5 promedio (µg/m³)", gridcolor=GRID_COLOR)
fig5.update_layout(barmode='group', legend=dict(orientation="h", y=1.1, x=0.02, xanchor="left"))
fig5.show()



### 7.5 Mapa interactivo

Incluye un menú desplegable con: filtro por categoría de riesgo (promedio general)
y dos vistas adicionales — **Entre Semana** y **Fin de Semana** — para comparar
directamente sobre el mapa.

> Nota: se usa el estilo base `carto-positron` (en vez de `open-street-map`) porque
> sus tiles sí incluyen las cabeceras CORS correctas: con `open-street-map` el mapa
> suele quedar en blanco al abrir el HTML directamente (protocolo `file://`).


In [ ]:

def add_map_trace(fig, sub, name, extra_label=""):
    fig.add_trace(go.Scattermap(
        lat=sub['lat'], lon=sub['lon'], mode='markers+text',
        marker=dict(size=18, color=[CAT_COLOR[c] for c in sub['Categoria']] if len(sub) else []),
        text=sub['Estacion'], textposition="top center", textfont=dict(size=10, color="#0F1B24"),
        name=name,
        customdata=np.stack([sub['Categoria'], sub['PM25_prom'].round(1), sub['count']], axis=-1) if len(sub) else None,
        hovertemplate=("<b>%{text}</b>" + extra_label +
                        "<br>Categoría: %{customdata[0]}<br>PM2.5 promedio: %{customdata[1]} µg/m³"
                        "<br>Registros válidos: %{customdata[2]}<extra></extra>") if len(sub) else "",
        visible=False,
    ))

fig4 = go.Figure()
add_map_trace(fig4, mapa_df, "Todas — Promedio general")
for cat in CAT_ORDER:
    add_map_trace(fig4, mapa_df[mapa_df['Categoria'] == cat], cat)
add_map_trace(fig4, mapa_tipodia[mapa_tipodia['Tipo_Dia'] == 'Entre Semana'], "Entre Semana",
              extra_label="<br><i>Entre Semana</i>")
add_map_trace(fig4, mapa_tipodia[mapa_tipodia['Tipo_Dia'] == 'Fin de Semana'], "Fin de Semana",
              extra_label="<br><i>Fin de Semana</i>")

n_traces = len(fig4.data)

def vis(idx):
    v = [False] * n_traces
    v[idx] = True
    return v

buttons = [dict(label="Todas las estaciones (promedio general)", method="update", args=[{"visible": vis(0)}])]
for i, cat in enumerate(CAT_ORDER):
    buttons.append(dict(label=f"{cat} (promedio general)", method="update", args=[{"visible": vis(1 + i)}]))
buttons.append(dict(label="Entre Semana (todas las estaciones)", method="update", args=[{"visible": vis(7)}]))
buttons.append(dict(label="Fin de Semana (todas las estaciones)", method="update", args=[{"visible": vis(8)}]))

fig4.data[0].visible = True

fig4.update_layout(
    map=dict(style="carto-positron", center=dict(lat=4.65, lon=-74.09), zoom=9.7),
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
                       x=0.01, y=0.99, xanchor="left", yanchor="top",
                       bgcolor="white", bordercolor="#D8DEE6")],
    height=580, margin=dict(l=0, r=0, t=50, b=0),
    title=dict(text="Mapa interactivo — filtra por categoría o compara Entre Semana / Fin de Semana",
               font=dict(size=17, color="#0F1B24"), x=0.02, xanchor="left"),
    font=FONT, showlegend=False,
)
fig4.show()



## 8. Ensamblar y exportar el dashboard HTML final

Genera un único archivo HTML autocontenido (usa Plotly vía CDN) con KPIs,
los 5 gráficos y dos tablas resumen, listo para abrir con doble clic.


In [ ]:

CONFIG = {"displaylogo": False, "responsive": True, "modeBarButtonsToRemove": ["lasso2d", "select2d"]}
chart_divs = {name: fig.to_html(full_html=False, include_plotlyjs=False, config=CONFIG)
              for name, fig in [("fig1", fig1), ("fig2", fig2), ("fig3", fig3), ("fig4", fig4), ("fig5", fig5)]}

fecha_min_s = pd.Timestamp(kpi['fecha_min']).strftime('%d %b %Y')
fecha_max_s = pd.Timestamp(kpi['fecha_max']).strftime('%d %b %Y')

# --- Tabla resumen por estación ---
rows = []
for _, r in est_stats.iterrows():
    color = CAT_COLOR[r['Categoria']]
    rows.append(f'''
    <tr>
      <td>{r['Estacion']}</td>
      <td class="num">{r['PM25_prom']:.1f}</td>
      <td class="num">{r['IBOCA_prom']:.1f}</td>
      <td><span class="tag" style="background:{color}22;color:{color};border:1px solid {color}66;">{r['Categoria']}</span></td>
      <td class="num">{int(r['count']):,}</td>
    </tr>''')
tabla_estaciones = "".join(rows)

table_html = f'''
<table class="data-table">
  <thead><tr><th>Estación</th><th class="num">PM2.5 promedio (µg/m³)</th>
  <th class="num">IBOCA promedio</th><th>Categoría</th><th class="num">Registros válidos</th></tr></thead>
  <tbody>{tabla_estaciones}</tbody>
</table>'''

pct_eliminadas = kpi['n_horas_sin_datos'] / kpi['n_total_horas'] * 100
calidad_table = f'''
<table class="data-table small">
  <thead><tr><th>Indicador</th><th class="num">Valor</th></tr></thead>
  <tbody>
    <tr><td>Marcas de tiempo totales en el período</td><td class="num">{kpi['n_total_horas']:,}</td></tr>
    <tr><td>Horas sin ningún dato reportado (eliminadas)</td><td class="num">{kpi['n_horas_sin_datos']:,} ({pct_eliminadas:.1f}%)</td></tr>
    <tr><td>Registros estación-hora válidos usados en el análisis</td><td class="num">{kpi['pct_validos']:.1f}% del total</td></tr>
  </tbody>
</table>'''

legend_items = "".join(
    f'<div class="legend-item"><span class="dot" style="background:{CAT_COLOR[c]}"></span>{c}</div>'
    for c in CAT_ORDER
)


In [ ]:

HTML = f'''<!doctype html>
<html lang="es"><head><meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>Calidad del Aire para Ciclistas y Peatones — Bogotá</title>
<script charset="utf-8" src="https://cdn.plot.ly/plotly-3.0.1.min.js"></script>
<style>
  :root {{ --bg:#F4F7FA; --card:#FFFFFF; --ink:#0F1B24; --sub:#55636F; --line:#E3E8EE; }}
  * {{ box-sizing: border-box; }}
  body {{ margin:0; padding:0; background:var(--bg); color:var(--ink); font-family:'Inter','Segoe UI',Arial,sans-serif; }}
  header {{ background: linear-gradient(120deg,#0F2540 0%, #163B63 55%, #1D4ED8 100%); color:#fff; padding:36px 40px 30px; }}
  header h1 {{ margin:0 0 6px; font-size:28px; font-weight:700; letter-spacing:-0.01em; }}
  header p {{ margin:0; color:#C9DAF2; font-size:14.5px; max-width:780px; line-height:1.5; }}
  .wrap {{ max-width:1240px; margin:0 auto; padding:0 24px; }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(190px, 1fr)); gap:14px; margin:-26px 0 28px; }}
  .kpi-card {{ background:var(--card); border:1px solid var(--line); border-radius:12px; padding:16px 18px; box-shadow:0 1px 2px rgba(15,27,36,0.04); }}
  .kpi-card .label {{ font-size:12px; color:var(--sub); margin-bottom:6px; }}
  .kpi-card .value {{ font-size:22px; font-weight:700; color:var(--ink); }}
  .kpi-card .value small {{ font-size:13px; font-weight:500; color:var(--sub); }}
  section {{ margin-bottom:26px; }}
  .card {{ background:var(--card); border:1px solid var(--line); border-radius:14px; padding:8px 18px 18px; box-shadow:0 1px 2px rgba(15,27,36,0.04); }}
  .card-title {{ font-size:15px; font-weight:600; color:var(--ink); padding:14px 4px 4px; }}
  .card-note {{ font-size:12.5px; color:var(--sub); padding:0 4px 10px; }}
  .legend-row {{ display:flex; flex-wrap:wrap; gap:14px; padding:4px 8px 4px; font-size:12.5px; color:var(--sub); }}
  .legend-item {{ display:flex; align-items:center; gap:6px; }}
  .legend-item .dot {{ width:10px; height:10px; border-radius:3px; display:inline-block; }}
  .two-col {{ display:grid; grid-template-columns:1.1fr 1fr; gap:18px; }}
  @media (max-width:980px) {{ .two-col {{ grid-template-columns:1fr; }} }}
  table.data-table {{ width:100%; border-collapse:collapse; font-size:13.5px; margin-top:4px; }}
  table.data-table th {{ text-align:left; font-weight:600; color:var(--sub); font-size:12px; padding:8px 10px; border-bottom:1px solid var(--line); }}
  table.data-table td {{ padding:7px 10px; border-bottom:1px solid #EFF2F6; }}
  table.data-table .num {{ text-align:right; font-variant-numeric:tabular-nums; }}
  .tag {{ display:inline-block; padding:2px 9px; border-radius:20px; font-size:12px; font-weight:600; }}
  footer {{ text-align:center; color:var(--sub); font-size:12.5px; padding:22px 0 40px; }}
</style></head>
<body>
<header><div class="wrap">
  <h1>🚴 Calidad del Aire para Ciclistas y Peatones — Bogotá</h1>
  <p>Análisis de concentración de PM2.5 en {kpi['n_estaciones']} estaciones de la Red de Monitoreo de
     Calidad del Aire de Bogotá (RMCAB), del {fecha_min_s} al {fecha_max_s} ({kpi['n_dias']} días).
     Las categorías de riesgo (colores) se calculan sobre el índice oficial IBOCA. Los datos fueron
     depurados: se excluyeron las horas sin ningún registro reportado por la red.</p>
</div></header>
<div class="wrap">
  <div class="kpi-grid">
    <div class="kpi-card"><div class="label">PM2.5 promedio de la ciudad</div>
      <div class="value">{kpi['pm25_prom']:.1f} <small>µg/m³ ({kpi['categoria_predominante']})</small></div></div>
    <div class="kpi-card"><div class="label">Estación más limpia</div>
      <div class="value">{kpi['estacion_mas_limpia']} <small>{kpi['estacion_mas_limpia_val']:.1f} µg/m³</small></div></div>
    <div class="kpi-card"><div class="label">Estación más contaminada</div>
      <div class="value">{kpi['estacion_mas_contaminada']} <small>{kpi['estacion_mas_contaminada_val']:.1f} µg/m³</small></div></div>
    <div class="kpi-card"><div class="label">Datos válidos usados</div><div class="value">{kpi['pct_validos']:.1f}%</div></div>
    <div class="kpi-card"><div class="label">Horas eliminadas (sin datos)</div>
      <div class="value">{kpi['n_horas_sin_datos']} <small>de {kpi['n_total_horas']}</small></div></div>
    <div class="kpi-card"><div class="label">Entre semana vs fin de semana</div>
      <div class="value">{kpi['entre_semana_n']:,} <small>/ {kpi['fin_semana_n']:,}</small></div></div>
  </div>
  <div class="legend-row">{legend_items}</div>
  <section><div class="card">{chart_divs['fig1']}</div></section>
  <section class="two-col"><div class="card">{chart_divs['fig2']}</div><div class="card">{chart_divs['fig3']}</div></section>
  <section><div class="card">{chart_divs['fig5']}</div></section>
  <section><div class="card">{chart_divs['fig4']}</div></section>
  <section class="two-col">
    <div class="card"><div class="card-title">Resumen por estación</div>
      <div class="card-note">Ordenado de menor a mayor PM2.5 promedio.</div>{table_html}</div>
    <div class="card"><div class="card-title">Calidad y limpieza de los datos</div>
      <div class="card-note">Horas sin ningún dato reportado por la red fueron eliminadas del conjunto de datos.</div>{calidad_table}</div>
  </section>
</div>
<footer>Fuente: Secretaría Distrital de Ambiente — RMCAB. Métrica principal: PM2.5 (µg/m³).
Categorías de riesgo sobre el índice oficial IBOCA. Límite diario PM2.5 en Colombia: 37 µg/m³ (Res. 2254 de 2017).</footer>
</body></html>'''

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(HTML)

print(f"Dashboard exportado a: {OUTPUT_HTML} ({len(HTML):,} bytes)")
